In [1]:
# Loại package không dùng trong bài nhưng đang ép NumPy >= 2
%pip uninstall -y tobler

# Khôi phục các dependency chung đúng với môi trường Colab hiện tại
%pip install -q --upgrade \
    "requests==2.32.4" \
    "packaging>=24.2" \
    "tenacity>=9,<10"

# Cài các package thực sự cần cho bài lab
# Không cài lại numpy/pandas vì Colab đã có sẵn
%pip install -q --upgrade-strategy only-if-needed \
    neo4j \
    pyarrow \
    openai \
    faiss-cpu \
    datasets \
    tqdm

Found existing installation: tobler 0.14.0
Uninstalling tobler-0.14.0:
  Successfully uninstalled tobler-0.14.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 84.4 MB/s eta 0:00:00


In [2]:
import os, re, json, time, random, hashlib, unicodedata
from pathlib import Path
from collections import defaultdict, Counter, deque
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from neo4j import GraphDatabase
import faiss

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 120)

def get_secret(name, default=None):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value is not None:
            return value
    except Exception:
        pass
    return os.environ.get(name, default)

NEO4J_URI = get_secret("NEO4J_URI", "")
NEO4J_USER = get_secret("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = get_secret("NEO4J_PASSWORD", "")
NEO4J_DATABASE = get_secret("NEO4J_DATABASE", "neo4j")

OPENAI_MODEL = get_secret("OPENAI_MODEL", "gpt-4o-mini")
OPENAI_EMBED_MODEL = get_secret("OPENAI_EMBED_MODEL", "text-embedding-3-small")
OPENAI_JUDGE_MODEL = get_secret("OPENAI_JUDGE_MODEL", OPENAI_MODEL)
OPENAI_API_KEY = get_secret("OPENAI_API_KEY", "")
HF_TOKEN = get_secret("HF_TOKEN", "")

DATA_PATH = "/content/hackernoon_subset.csv"
LAB_MAX_ARTICLES = 1500
LAB_MAX_CHUNKS = 3000
EXTRACTION_MAX_CHUNKS = 400
CHUNK_WORDS = 220
CHUNK_OVERLAP_WORDS = 40

In [3]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

# print("Token exists:", token is not None)
# print("Token prefix OK:", token.startswith("hf_") if token else False)

In [4]:
#@title 1.3 — Stream HackerNoon dataset -> CSV
import csv
import os
from datasets import load_dataset
from tqdm.auto import tqdm

DATASET_NAME = "HackerNoon/tech-company-news-data-dump"
OUTPUT_CSV = "/content/hackernoon_subset.csv"

# Giới hạn cho bản lab. Có thể tăng sau buổi học.
LIMIT_ROWS = 5000
LIMIT_MB = 300

# True  -> progress/dừng ưu tiên theo MB
# False -> progress theo rows; vẫn có hard-stop LIMIT_ROWS
PRIORITIZE_MB = True


# Đọc từ Colab Secrets qua get_secret() ở cell config.
if not HF_TOKEN:
    raise ValueError(
        "Thiếu HF_TOKEN. Hãy thêm Hugging Face Access Token vào Colab Secrets với tên HF_TOKEN."
    )

print("Đang kết nối luồng dữ liệu (streaming)...")

try:
    dataset = load_dataset(
        DATASET_NAME,
        split="train",
        streaming=True,
        token=HF_TOKEN,
    )
    iterator = iter(dataset)

    first_row = next(iterator)
    headers = list(first_row.keys())

    print(f"Đang ghi dữ liệu vào: {OUTPUT_CSV}")

    rows_written = 0
    total_progress = LIMIT_MB if PRIORITIZE_MB else LIMIT_ROWS
    unit_progress = "MB" if PRIORITIZE_MB else "row"

    with open(OUTPUT_CSV, mode="w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=headers, extrasaction="ignore")
        writer.writeheader()
        writer.writerow(first_row)
        rows_written += 1

        # Flush để kích thước file phản ánh dữ liệu vừa ghi.
        f.flush()
        file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)

        with tqdm(
            total=total_progress,
            desc=f"Đang tải ({unit_progress})",
            unit=unit_progress,
        ) as pbar:
            if PRIORITIZE_MB:
                pbar.n = min(file_size_mb, LIMIT_MB)
                pbar.refresh()
            else:
                pbar.update(1)

            for row in iterator:
                writer.writerow(row)
                rows_written += 1

                # Kiểm tra dung lượng định kỳ để giảm overhead I/O.
                # Khi gần LIMIT_MB, kiểm tra mỗi row để dừng sát ngưỡng hơn.
                should_check_size = (
                    PRIORITIZE_MB
                    and (
                        rows_written % 100 == 0
                        or file_size_mb >= LIMIT_MB * 0.95
                    )
                )

                if should_check_size:
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    pbar.n = min(round(file_size_mb, 2), LIMIT_MB)
                    pbar.refresh()
                elif not PRIORITIZE_MB:
                    pbar.update(1)

                # Hard-stop theo MB nếu đang ưu tiên dung lượng.
                if PRIORITIZE_MB and file_size_mb >= LIMIT_MB:
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn dung lượng: "
                        f"{file_size_mb:.2f} MB "
                        f"(Tổng: {rows_written:,} dòng)"
                    )
                    break

                # Hard-stop theo số dòng trong mọi chế độ.
                if rows_written >= LIMIT_ROWS:
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn số dòng: "
                        f"{rows_written:,} dòng "
                        f"(Dung lượng: {file_size_mb:.2f} MB)"
                    )
                    break

        f.flush()

    final_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
    print(
        f"✅ Hoàn thành: {os.path.abspath(OUTPUT_CSV)}\n"
        f"   Rows: {rows_written:,}\n"
        f"   Size: {final_size_mb:.2f} MB"
    )

    # Đồng bộ đường dẫn cho cell loader tiếp theo.
    DATA_PATH = OUTPUT_CSV

except StopIteration:
    raise RuntimeError("Dataset stream rỗng: không lấy được dòng đầu tiên.")
except Exception as e:
    print(f"\n❌ Có lỗi xảy ra: {e}")
    print(
        "Kiểm tra: (1) HF_TOKEN, (2) quyền Agree/Access trên Hugging Face, "
        "(3) kết nối mạng của Colab."
    )
    raise


Đang kết nối luồng dữ liệu (streaming)...


README.md:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

Đang ghi dữ liệu vào: /content/hackernoon_subset.csv


Đang tải (MB):   0%|          | 0/300 [00:00<?, ?MB/s]


[DỪNG] Đã đạt giới hạn số dòng: 5,000 dòng (Dung lượng: 2.92 MB)
✅ Hoàn thành: /content/hackernoon_subset.csv
   Rows: 5,000
   Size: 2.92 MB


In [5]:
from google.colab import userdata
NEO4J_USER = userdata.get('NEO4J_USER')
NEO4J_PASSWORD = userdata.get('NEO4J_PASSWORD')
NEO4J_URI = userdata.get('NEO4J_URI')
NEO4J_DATABASE = userdata.get('NEO4J_DATABASE')

In [6]:
driver = None

def connect_neo4j():
    global driver
    if not NEO4J_URI or not NEO4J_PASSWORD:
        raise ValueError("Thiếu Neo4j secrets.")
    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USER, NEO4J_PASSWORD),
    )
    driver.verify_connectivity()
    print("✅ Neo4j connected.")

def run_cypher(query, **params):
    if driver is None:
        raise RuntimeError("Hãy chạy connect_neo4j() trước.")
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query, **params)
        rows = [r.data() for r in result]
        result.consume()
    return rows

def setup_graph_schema():
    for stmt in [
        """
        CREATE CONSTRAINT entity_id IF NOT EXISTS
        FOR (n:Entity) REQUIRE n.id IS UNIQUE
        """,
        """
        CREATE INDEX entity_name_norm IF NOT EXISTS
        FOR (n:Entity) ON (n.name_norm)
        """,
        """
        CREATE INDEX company_name_norm IF NOT EXISTS
        FOR (n:Company) ON (n.name_norm)
        """,
        """
        CREATE INDEX person_name_norm IF NOT EXISTS
        FOR (n:Person) ON (n.name_norm)
        """,
        """
        CREATE INDEX technology_name_norm IF NOT EXISTS
        FOR (n:Technology) ON (n.name_norm)
        """,
    ]:
        run_cypher(stmt)
    print("✅ Schema ready.")

connect_neo4j()
setup_graph_schema()


✅ Neo4j connected.
✅ Schema ready.


In [7]:
def norm_space(x):
    return re.sub(r"\s+", " ", str(x or "")).strip()

def sha1(x):
    return hashlib.sha1(str(x).encode("utf-8", errors="ignore")).hexdigest()

def pick_col(df, candidates, required=True):
    lookup = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lookup:
            return lookup[c.lower()]
    if required:
        raise KeyError(f"Missing one of columns: {candidates}")
    return None

def load_news(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".jsonl", ".ndjson"}:
        return pd.read_json(path, lines=True)
    if path.suffix.lower() == ".json":
        return pd.read_json(path)
    if path.suffix.lower() in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported: {path.suffix}")

def standardize_news(raw):
    text_col = pick_col(raw, ["text", "content", "article", "body", "story", "description"])
    title_col = pick_col(raw, ["title", "headline"], required=False)
    date_col = pick_col(raw, ["published_date", "date", "published_at", "created_at"], required=False)
    id_col = pick_col(raw, ["id", "article_id", "story_id", "uuid"], required=False)

    df = pd.DataFrame()
    df["text"] = raw[text_col].fillna("").map(norm_space)
    df["title"] = raw[title_col].fillna("").map(norm_space) if title_col else ""

    if date_col:
        df["published_date"] = (
            pd.to_datetime(raw[date_col], errors="coerce", utc=True)
            .dt.strftime("%Y-%m-%d")
            .fillna("")
        )
    else:
        df["published_date"] = ""

    if id_col:
        df["article_id"] = raw[id_col].astype(str)
    else:
        df["article_id"] = [
            sha1(f"{t}\n{x}")[:20] for t, x in zip(df["title"], df["text"])
        ]

    df = df[df["text"].str.len() >= 80].copy()
    df["dedup_key"] = [
        sha1(norm_space(f"{t}\n{x}").lower())
        for t, x in zip(df["title"], df["text"])
    ]
    before = len(df)
    df = df.drop_duplicates("dedup_key").drop(columns="dedup_key").reset_index(drop=True)
    print(f"Exact dedup: {before:,} -> {len(df):,}")

    if LAB_MAX_ARTICLES and len(df) > LAB_MAX_ARTICLES:
        df = df.sample(LAB_MAX_ARTICLES, random_state=SEED).sort_index().reset_index(drop=True)
    return df

def chunk_text(text, size=220, overlap=40):
    words = norm_space(text).split()
    step = max(1, size - overlap)
    out = []
    for start in range(0, len(words), step):
        part = words[start:start+size]
        if not part:
            break
        out.append(" ".join(part))
        if start + size >= len(words):
            break
    return out

def build_chunks(news_df):
    rows = []
    for r in tqdm(news_df.itertuples(index=False), total=len(news_df), desc="Chunking"):
        for i, text in enumerate(chunk_text(r.text, CHUNK_WORDS, CHUNK_OVERLAP_WORDS)):
            rows.append({
                "chunk_id": f"{r.article_id}::c{i:04d}",
                "article_id": r.article_id,
                "title": r.title,
                "published_date": r.published_date,
                "text": text,
            })
            if LAB_MAX_CHUNKS and len(rows) >= LAB_MAX_CHUNKS:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)

raw_df = load_news(DATA_PATH)
display(raw_df.head())
news_df = standardize_news(raw_df)
chunks_df = build_chunks(news_df)
display(chunks_df.head())

,companyName,companyUrl,published_at,url,title,main_image,description
0,01Synergy,https://hackernoon.com/company/01synergy,2023-05-16 02:09:00,https://www.businesswire.com/news/home/20230515005855/en/onsemi-and-Sineng-Electric-Spearhead-the-Development-of-Sus...,onsemi and Sineng Electric Spearhead the Development of Sustainable Energy Applications,https://firebasestorage.googleapis.com/v0/b/hackernoon-app.appspot.com/o/images%2Fimageedit_25_7084755369.gif?alt=me...,(Nasdaq: ON) a leader in intelligent power and sensing technologies today announced that Sineng Electric will integr...
1,01Synergy,https://hackernoon.com/company/01synergy,2023-05-02 00:07:00,https://elkodaily.com/news/local/adobe-student-receives-national-information-and-technology-award/article_ad2d9924-e...,Adobe student receives national Information and Technology award,https://firebasestorage.googleapis.com/v0/b/hackernoon-app.appspot.com/o/images%2Fimageedit_25_7084755369.gif?alt=me...,ELKO — An eighth grader at Adobe Middle School is one of 34 middle school aged girls in Nevada to be recognized by t...
2,01Synergy,https://hackernoon.com/company/01synergy,2023-05-01 22:22:00,https://www.aei.org/technology-and-innovation/modernizing-state-services-harnessing-technology-for-enhanced-public-s...,Modernizing State Services: Harnessing Technology for Enhanced Public Service Delivery,https://firebasestorage.googleapis.com/v0/b/hackernoon-app.appspot.com/o/images%2Fimageedit_25_7084755369.gif?alt=me...,To deliver 21st-century government services Governors and cabinet members need leaders with technology expertise to ...
3,01Synergy,https://hackernoon.com/company/01synergy,2023-05-02 13:12:00,https://www.crn.com/news/managed-services/terry-richardson-on-why-he-left-amd-greenpages-technology-chops-and-the-ai...,Terry Richardson On Why He Left AMD GreenPages’ Technology Chops And The AI Opportunity,https://firebasestorage.googleapis.com/v0/b/hackernoon-app.appspot.com/o/images%2Fimageedit_25_7084755369.gif?alt=me...,In February GreenPages acquired Toronto-based Zanaris an IT automation cloud and DevOps services firm ... Steve Burk...
4,01Synergy,https://hackernoon.com/company/01synergy,2023-05-15 20:01:00,https://www.benzinga.com/pressreleases/23/05/32404522/synex-renewable-energy-corporation-formerly-synex-internationa...,Synex Renewable Energy Corporation (Formerly Synex International Inc.) Third Quarter of Fiscal 2023,https://firebasestorage.googleapis.com/v0/b/hackernoon-app.appspot.com/o/images%2Fimageedit_25_7084755369.gif?alt=me...,The conference will bring together growth oriented publicly traded clean energy and technology companies ... up to 4...


Exact dedup: 2,675 -> 2,105


Chunking:   0%|          | 0/1500 [00:00<?, ?it/s]

,chunk_id,article_id,title,published_date,text
0,1a05beb7aa3071be6fd7::c0000,1a05beb7aa3071be6fd7,onsemi and Sineng Electric Spearhead the Development of Sustainable Energy Applications,2023-05-16,(Nasdaq: ON) a leader in intelligent power and sensing technologies today announced that Sineng Electric will integr...
1,8e922bc62b578e73e815::c0000,8e922bc62b578e73e815,Modernizing State Services: Harnessing Technology for Enhanced Public Service Delivery,2023-05-01,To deliver 21st-century government services Governors and cabinet members need leaders with technology expertise to ...
2,4bd7afdba71243b0dbcd::c0000,4bd7afdba71243b0dbcd,Terry Richardson On Why He Left AMD GreenPages’ Technology Chops And The AI Opportunity,2023-05-02,In February GreenPages acquired Toronto-based Zanaris an IT automation cloud and DevOps services firm ... Steve Burk...
3,6633c15d86f5e81f47b8::c0000,6633c15d86f5e81f47b8,5 Kubernetes technology vendors hot right now,2023-02-28,Kubernetes is a technology that has created a whole new ecosystem around itself and it is now a key plank in the Dev...
4,4ce72a4490a6a618e2d5::c0000,4ce72a4490a6a618e2d5,Bachelor of Science in Health Information Management,2023-08-16,Health information management (HIM) is a diverse yet evolving field that incorporates medicine management finance in...


In [8]:
# OpenAI configuration is loaded in the setup cell above.

In [9]:
from openai import OpenAI
openai_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

def parse_json_object(text):
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    a, b = text.find("{"), text.rfind("}")
    if a < 0 or b <= a:
        raise ValueError("No JSON object found.")
    return json.loads(text[a:b+1])

def openai_chat(messages, model=None, json_mode=False, max_retries=4):
    if openai_client is None:
        raise RuntimeError("Thiếu OPENAI_API_KEY.")
    model = model or OPENAI_MODEL
    if not model:
        raise RuntimeError("Thiếu OPENAI_MODEL.")

    last = None
    for attempt in range(max_retries):
        try:
            kwargs = {
                "model": model,
                "messages": messages,
                "temperature": 0.0,
            }
            if json_mode:
                kwargs["response_format"] = {"type": "json_object"}

            resp = openai_client.chat.completions.create(**kwargs)
            usage = {}
            if getattr(resp, "usage", None):
                usage = {
                    "prompt_tokens": getattr(resp.usage, "prompt_tokens", None),
                    "completion_tokens": getattr(resp.usage, "completion_tokens", None),
                    "total_tokens": getattr(resp.usage, "total_tokens", None),
                }
            return resp.choices[0].message.content, usage
        except Exception as e:
            last = e
            if attempt == max_retries - 1:
                break
            time.sleep(min(20, 2**attempt + random.random()))
    raise RuntimeError(last)

def openai_json(system, user, model=None):
    text, usage = openai_chat(
        [{"role": "system", "content": system},
         {"role": "user", "content": user}],
        model=model,
        json_mode=True,
    )
    return parse_json_object(text), usage


In [10]:
#@title 1.7 — Coreference resolution theo batch
COREF_SYSTEM = """
You are a conservative coreference-resolution component for a knowledge-graph pipeline.
Resolve pronouns and generic references only when the antecedent is clearly supported in the same chunk.
Never invent facts. Preserve dates, numbers, tickers and product names.
Return strict JSON only.
""".strip()

def resolve_coref_batch(batch_df):
    payload = [{"chunk_id": r.chunk_id, "text": r.text}
               for r in batch_df.itertuples(index=False)]

    prompt = f"""
Resolve coreferences.

Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "resolved_text": "...",
      "unresolved_mentions": ["..."]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()

    obj, usage = openai_json(COREF_SYSTEM, prompt)
    by_id = {x.get("chunk_id"): x for x in obj.get("items", [])}

    rows = []
    for r in batch_df.itertuples(index=False):
        item = by_id.get(r.chunk_id, {})
        rows.append({
            "chunk_id": r.chunk_id,
            "resolved_text": norm_space(item.get("resolved_text") or r.text),
            "unresolved_mentions": item.get("unresolved_mentions", []),
        })
    return pd.DataFrame(rows), usage

def run_coref(chunks_subset, batch_size=5):
    out = []
    for start in tqdm(range(0, len(chunks_subset), batch_size), desc="Coref"):
        batch = chunks_subset.iloc[start:start+batch_size]
        try:
            df, _ = resolve_coref_batch(batch)
        except Exception:
            print("error")
            df = pd.DataFrame({
                "chunk_id": batch["chunk_id"].tolist(),
                "resolved_text": batch["text"].tolist(),
                "unresolved_mentions": [["COREF_BATCH_FAILED"] for _ in range(len(batch))],
            })
        out.append(df)
    return pd.concat(out, ignore_index=True)

extraction_source = chunks_df.head(EXTRACTION_MAX_CHUNKS).copy()
coref_df = run_coref(extraction_source)
extraction_source = extraction_source.merge(coref_df, on="chunk_id", how="left")


Coref:   0%|          | 0/80 [00:00<?, ?it/s]

In [11]:
ALLOWED_NODE_TYPES = {"Company", "Person", "Technology"}
ALLOWED_RELATIONS = {
    "ACQUIRED", "DEVELOPED", "INVESTED_IN", "FOUNDED",
    "WORKED_AT", "PARTNERED_WITH", "USES", "LEADS"
}

EXTRACT_SYSTEM = f"""
Extract a high-precision knowledge graph from tech-news text.
Allowed node types: {sorted(ALLOWED_NODE_TYPES)}
Allowed relations: {sorted(ALLOWED_RELATIONS)}
Use only explicitly supported facts. Prefer precision over recall.
Every relation needs short evidence. Return strict JSON only.
""".strip()

def extract_batch(batch_df):
    payload = [{
        "chunk_id": r.chunk_id,
        "published_date": r.published_date,
        "text": getattr(r, "resolved_text", None) or r.text,
    } for r in batch_df.itertuples(index=False)]

    prompt = f"""
Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "relations": [
        {{
          "source": "...",
          "source_type": "Company|Person|Technology",
          "relation": "ALLOWED_RELATION",
          "target": "...",
          "target_type": "Company|Person|Technology",
          "evidence": "...",
          "confidence": 0.0
        }}
      ]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()
    return openai_json(EXTRACT_SYSTEM, prompt)

def run_extraction(source_df, batch_size=4):
    meta = source_df.set_index("chunk_id")["published_date"].to_dict()
    triples, errors = [], []

    for start in tqdm(range(0, len(source_df), batch_size), desc="NER+RE"):
        batch = source_df.iloc[start:start+batch_size]
        try:
            obj, _ = extract_batch(batch)
        except Exception as e:
            print(e)
            errors.append({"start": start, "error": str(e)})
            continue

        for item in obj.get("items", []):
            cid = item.get("chunk_id")
            if cid not in meta:
                continue
            for x in item.get("relations", []):
                s, t = norm_space(x.get("source")), norm_space(x.get("target"))
                st, tt, rel = x.get("source_type"), x.get("target_type"), x.get("relation")
                if not s or not t:
                    continue
                if st not in ALLOWED_NODE_TYPES or tt not in ALLOWED_NODE_TYPES:
                    continue
                if rel not in ALLOWED_RELATIONS:
                    continue
                triples.append({
                    "source_raw": s,
                    "source_type": st,
                    "relation": rel,
                    "target_raw": t,
                    "target_type": tt,
                    "source_chunk_id": cid,
                    "published_date": meta[cid] or "",
                    "evidence": norm_space(x.get("evidence")),
                    "confidence": float(x.get("confidence") or 0.0),
                })

    return pd.DataFrame(triples), pd.DataFrame(errors)

raw_triples_df, extraction_errors_df = run_extraction(extraction_source)
display(raw_triples_df.head())


NER+RE:   0%|          | 0/100 [00:00<?, ?it/s]

,source_raw,source_type,relation,target_raw,target_type,source_chunk_id,published_date,evidence,confidence
0,GreenPages,Company,ACQUIRED,Zanaris,Company,4bd7afdba71243b0dbcd::c0000,2023-05-02,"GreenPages acquired Toronto-based Zanaris, an IT automation cloud and DevOps services firm",1.0
1,Sineng Electric,Company,PARTNERED_WITH,onsemi EliteSiC,Technology,1a05beb7aa3071be6fd7::c0000,2023-05-16,Sineng Electric will integrate onsemi EliteSiC silic,1.0
2,Kubernetes,Technology,DEVELOPED,DevOps movement,Technology,6633c15d86f5e81f47b8::c0000,2023-02-28,Kubernetes is now a key plank in the DevOps movement,1.0
3,Stewart Information Services Corporation,Company,FOUNDED,property and casualty insurance industry,Technology,e6f458adb39e8a23889e::c0000,2023-08-14,Stewart Information Services Corporation is in the property and casualty insurance industry.,0.9
4,Mark Leary,Person,WORKED_AT,IDC,Company,24e2333c327a81eca0f6::c0000,2023-03-03,"Mark Leary, research director for network analytics and automation at IDC.",0.9


In [12]:
#@title 2.2 — Entity resolution
CORP_SUFFIXES = {"inc","incorporated","corp","corporation","ltd","limited","llc","plc","co","company"}
MANUAL_ALIASES = {
    "msft": "Microsoft",
    "microsoft corp": "Microsoft",
    "microsoft corporation": "Microsoft",
    "goog": "Google",
    "googl": "Google",
    "google llc": "Google",
    "meta platforms": "Meta",
    "meta platforms inc": "Meta",
    "aapl": "Apple",
    "apple inc": "Apple",
}

def norm_entity(name):
    s = unicodedata.normalize("NFKC", norm_space(name)).lower()
    s = re.sub(r"[^\w\s\-\.]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def strip_suffix(name):
    toks = norm_entity(name).replace(".", "").split()
    while toks and toks[-1] in CORP_SUFFIXES:
        toks.pop()
    return " ".join(toks)

def merge_guard(a, b):
    na, nb = strip_suffix(a), strip_suffix(b)
    if na == nb:
        return True
    return SequenceMatcher(None, na, nb).ratio() >= 0.72

embedding_cache = {}

def get_embedder():
    if openai_client is None:
        raise RuntimeError("Thiếu OPENAI_API_KEY.")
    return openai_client.embeddings
def embed_texts(texts, batch_size=128):
    texts = [str(x or "") for x in texts]
    missing = [x for x in texts if x not in embedding_cache]
    for start in range(0, len(missing), batch_size):
        batch = missing[start:start + batch_size]
        response = get_embedder().create(model=OPENAI_EMBED_MODEL, input=batch)
        for text, item in zip(batch, sorted(response.data, key=lambda x: x.index)):
            vector = np.asarray(item.embedding, dtype="float32")
            norm = np.linalg.norm(vector)
            embedding_cache[text] = vector / norm if norm else vector
    return np.vstack([embedding_cache[x] for x in texts]).astype("float32")

class UF:
    def __init__(self, n):
        self.p = list(range(n))
    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])
        return self.p[x]
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b:
            self.p[b] = a

def build_resolution_map(raw_triples_df, threshold=0.90, top_k=5):
    mentions = []
    for r in raw_triples_df.itertuples(index=False):
        mentions += [(r.source_type, r.source_raw), (r.target_type, r.target_raw)]

    counts = Counter((t, norm_entity(n)) for t, n in mentions)
    display_name = {}
    for t, n in mentions:
        display_name.setdefault((t, norm_entity(n)), n)

    mapping, audit = {}, []

    for key in counts:
        t, norm = key
        if norm in MANUAL_ALIASES:
            mapping[key] = MANUAL_ALIASES[norm]
            audit.append({
                "type": t, "left": display_name[key],
                "right": MANUAL_ALIASES[norm],
                "similarity": 1.0, "decision": "MERGE_MANUAL"
            })

    for typ in sorted(ALLOWED_NODE_TYPES):
        keys = [k for k in counts if k[0] == typ and k not in mapping]
        if not keys:
            continue
        names = [display_name[k] for k in keys]
        vecs = embed_texts(
            names, batch_size=128
        ).astype("float32")

        index = faiss.IndexFlatIP(vecs.shape[1])
        index.add(vecs)
        sims, nbrs = index.search(vecs, min(top_k, len(names)))
        uf = UF(len(names))

        for i in range(len(names)):
            for score, j in zip(sims[i], nbrs[i]):
                if j < 0 or i >= j or float(score) < threshold:
                    continue
                ok = merge_guard(names[i], names[j])
                audit.append({
                    "type": typ, "left": names[i], "right": names[j],
                    "similarity": float(score),
                    "decision": "MERGE_VECTOR" if ok else "REJECT_GUARD"
                })
                if ok:
                    uf.union(i, j)

        groups = defaultdict(list)
        for i in range(len(names)):
            groups[uf.find(i)].append(i)

        for idxs in groups.values():
            best = sorted(
                idxs,
                key=lambda i: (-counts[keys[i]], len(names[i]), names[i].lower())
            )[0]
            canonical = names[best]
            for i in idxs:
                mapping[keys[i]] = canonical

    for key in counts:
        mapping.setdefault(key, display_name[key])

    return mapping, pd.DataFrame(audit)

def canonicalize_triples(raw_df, mapping):
    df = raw_df.copy()
    def canon(name, typ):
        n = norm_entity(name)
        return mapping.get((typ, n), MANUAL_ALIASES.get(n, name))

    df["source_name"] = [canon(n,t) for n,t in zip(df.source_raw, df.source_type)]
    df["target_name"] = [canon(n,t) for n,t in zip(df.target_raw, df.target_type)]
    df["source_name_norm"] = df.source_name.map(norm_entity)
    df["target_name_norm"] = df.target_name.map(norm_entity)
    df["source_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.source_type, df.source_name_norm)]
    df["target_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.target_type, df.target_name_norm)]
    return df[df.source_id != df.target_id].reset_index(drop=True)

entity_map, entity_resolution_audit_df = build_resolution_map(raw_triples_df)
triples_df = canonicalize_triples(raw_triples_df, entity_map)
display(entity_resolution_audit_df.head(20))


,type,left,right,similarity,decision
0,Company,Information Services Group,Information Services Group Inc.,0.935348,MERGE_VECTOR
1,Company,Amazon Web Services (AWS),Amazon Web Services,0.941637,MERGE_VECTOR
2,Company,Fidelity National Information Services,Fidelity National Information Services Inc.,0.982646,MERGE_VECTOR
3,Company,Biz Technology Solutions,Biz Technology Solutions Inc.,0.951118,MERGE_VECTOR
4,Technology,X90A,X90,0.932864,MERGE_VECTOR


In [22]:
def build_nodes(triples_df):
    rows = []
    for r in triples_df.itertuples(index=False):
        rows += [
            {"id":r.source_id,"name":r.source_name,"name_norm":r.source_name_norm,"type":r.source_type,"alias":r.source_raw},
            {"id":r.target_id,"name":r.target_name,"name_norm":r.target_name_norm,"type":r.target_type,"alias":r.target_raw},
        ]
    tmp = pd.DataFrame(rows)
    if tmp.empty:
        return tmp

    out = []
    for (node_id,name,name_norm,typ), g in tmp.groupby(["id","name","name_norm","type"]):
        aliases = sorted(set(g["alias"].map(norm_space)))
        out.append({
            "id":node_id, "name":name, "name_norm":name_norm, "type":typ,
            "aliases":aliases,
            "aliases_norm":sorted(set(norm_entity(x) for x in aliases))
        })
    return pd.DataFrame(out)

def batches(records, size=1000):
    for i in range(0, len(records), size):
        yield records[i:i+size]

def bulk_insert_nodes(nodes_df, batch_size=1000):
    for typ in sorted(ALLOWED_NODE_TYPES):
        part = nodes_df[nodes_df.type == typ]
        if part.empty:
            continue
        query = f"""
        UNWIND $rows AS row
        MERGE (n:Entity {{id: row.id}})
        SET n:{typ},
            n.name=row.name,
            n.name_norm=row.name_norm,
            n.entity_type=row.type,
            n.aliases=row.aliases,
            n.aliases_norm=row.aliases_norm
        """
        for b in batches(part.to_dict("records"), batch_size):
            run_cypher(query, rows=b)

def bulk_insert_edges(triples_df, batch_size=1000):
    required = {"source_chunk_id","published_date"}
    if not required.issubset(triples_df.columns):
        raise ValueError("Missing edge provenance.")

    for rel in sorted(ALLOWED_RELATIONS):
        part = triples_df[triples_df.relation == rel]
        if part.empty:
            continue

        query = f"""
        UNWIND $rows AS row
        MATCH (s:Entity {{id: row.source_id}})
        MATCH (t:Entity {{id: row.target_id}})
        MERGE (s)-[r:{rel} {{source_chunk_id: row.source_chunk_id}}]->(t)
        SET r.published_date=row.published_date,
            r.evidence=row.evidence,
            r.confidence=row.confidence
        """

        cols = ["source_id","target_id","source_chunk_id","published_date","evidence","confidence"]
        for b in batches(part[cols].to_dict("records"), batch_size):
            run_cypher(query, rows=b)

nodes_df = build_nodes(triples_df)
bulk_insert_nodes(nodes_df)
bulk_insert_edges(triples_df)


In [23]:
def graph_checks():
    invalid = run_cypher("""
    MATCH ()-[r]->()
    WHERE r.source_chunk_id IS NULL OR r.published_date IS NULL
    RETURN count(r) AS n
    """)[0]["n"]

    counts = {
        "nodes": run_cypher("MATCH (n:Entity) RETURN count(n) AS n")[0]["n"],
        "edges": run_cypher("MATCH ()-[r]->() RETURN count(r) AS n")[0]["n"],
        "invalid_provenance_edges": invalid,
    }
    print(counts)
    assert invalid == 0

    top = pd.DataFrame(run_cypher("""
    MATCH (n:Entity)
    OPTIONAL MATCH (n)-[r]-()
    WITH n, count(r) AS degree
    RETURN n.id AS id, n.name AS name, n.entity_type AS type, degree
    ORDER BY degree DESC LIMIT 15
    """))
    display(top)
    return counts, top

graph_counts, top_degree_df = graph_checks()


{'nodes': 286, 'edges': 162, 'invalid_provenance_edges': 0}


,id,name,type,degree
0,cc9c6ee3857729e221d3f6de,ServiceNow,Company,3
1,482e949a0b6c21ca2b8da03e,Infineon,Company,3
2,b2d4ccacccbb87fe51c0d279,Amazon Web Services (AWS),Company,3
3,b4fe88d52bfdb340ad30a63c,video streaming services,Technology,3
4,9960a1ca5dd13b4f18ca1fb7,New Charter Technologies,Company,3
5,89558874b295d1e5bb9dfb44,Darling Ingredients,Company,2
6,773eeb9b7cc008bff365fcdd,OpenAI,Company,2
7,9288e25745f74c90f2ba0187,Broadridge Financial Solutions,Company,2
8,0cf2f2945e5f6dc338954501,SoftBank,Company,2
9,26fe01fbff18a0ced2a65600,Infosys,Company,2


In [24]:
flat_index = None
flat_store = None
entity_match_vectors = None
entity_match_store = None

def build_flat_index(chunks_df):
    global flat_index, flat_store
    vecs = embed_texts(
        chunks_df.text.fillna("").tolist(),
        batch_size=128
    ).astype("float32")

    flat_index = faiss.IndexFlatIP(vecs.shape[1])
    flat_index.add(vecs)
    flat_store = chunks_df.reset_index(drop=True).copy()
    print("Flat vectors:", flat_index.ntotal)

def retrieve_flat_context(query, k=6):
    qv = embed_texts(
        [query]
    ).astype("float32")
    scores, ids = flat_index.search(qv, min(k, flat_index.ntotal))

    rows = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0:
            continue
        r = flat_store.iloc[int(idx)]
        rows.append({
            "score":float(score), "chunk_id":r.chunk_id,
            "published_date":r.published_date, "text":r.text
        })

    df = pd.DataFrame(rows)
    context = "\n\n".join(
        f"[chunk_id={r.chunk_id} | date={r.published_date} | score={r.score:.3f}]\n{r.text}"
        for r in df.itertuples(index=False)
    )
    return context, df

build_flat_index(chunks_df)


Flat vectors: 1500


In [25]:
SEED_SYSTEM = """
Extract useful seed entities for graph retrieval.
Allowed types: Company, Person, Technology.
Do not answer the question. Return strict JSON only.
""".strip()

def extract_seeds(query):
    obj, _ = openai_json(SEED_SYSTEM, f"""
Question: {query}
Return {{"seeds":[{{"name":"...","type":"Company|Person|Technology|null"}}]}}
""")
    return [
        {"name":norm_space(x.get("name")),
         "type":x.get("type") if x.get("type") in ALLOWED_NODE_TYPES else None}
        for x in obj.get("seeds", [])
        if norm_space(x.get("name"))
    ]

def build_entity_matcher(nodes_df):
    global entity_match_vectors, entity_match_store
    entity_match_store = nodes_df.reset_index(drop=True).copy()
    entity_match_vectors = embed_texts(
        entity_match_store.name.tolist(),
        batch_size=128
    ).astype("float32")

def match_seeds(query, fuzzy_threshold=0.66):
    matched = []
    for seed in extract_seeds(query):
        exact = run_cypher("""
        MATCH (n:Entity)
        WHERE (n.name_norm=$name OR $name IN coalesce(n.aliases_norm,[]))
          AND ($typ IS NULL OR n.entity_type=$typ)
        RETURN n.id AS id, n.name AS name, n.entity_type AS type
        LIMIT 5
        """, name=norm_entity(seed["name"]), typ=seed["type"])

        if exact:
            matched += exact
            continue

        if entity_match_vectors is None:
            continue

        mask = np.ones(len(entity_match_store), dtype=bool)
        if seed["type"]:
            mask = entity_match_store.type.eq(seed["type"]).to_numpy()
        idxs = np.flatnonzero(mask)
        if not len(idxs):
            continue

        qv = embed_texts(
            [seed["name"]]
        ).astype("float32")[0]
        sims = entity_match_vectors[idxs] @ qv
        j = int(np.argmax(sims))
        if float(sims[j]) >= fuzzy_threshold:
            r = entity_match_store.iloc[int(idxs[j])]
            matched.append({"id":r.id,"name":r.name,"type":r.type})

    return list({x["id"]: x for x in matched}.values())

build_entity_matcher(nodes_df)


In [26]:
SUPER_NODE_DEGREE = 100
SUPER_NODE_EDGE_CAP = 50
GLOBAL_EDGE_CAP = 250
MAX_GRAPH_CONTEXT_CHARS = 14000

def node_degree(node_id):
    return int(run_cypher("""
    MATCH (n:Entity {id:$id})
    OPTIONAL MATCH (n)-[r]-()
    RETURN count(r) AS degree
    """, id=node_id)[0]["degree"])

def recent_edges(node_id, limit):
    return run_cypher("""
    MATCH (n:Entity {id:$id})
    MATCH (n)-[r]-(m:Entity)
    RETURN
      startNode(r).id AS source_id,
      startNode(r).name AS source_name,
      startNode(r).entity_type AS source_type,
      type(r) AS relation,
      endNode(r).id AS target_id,
      endNode(r).name AS target_name,
      endNode(r).entity_type AS target_type,
      r.source_chunk_id AS source_chunk_id,
      r.published_date AS published_date,
      r.evidence AS evidence,
      m.id AS neighbor_id
    ORDER BY coalesce(r.published_date,'') DESC
    LIMIT $limit
    """, id=node_id, limit=int(limit))

def textualize(edges):
    edges = sorted(edges, key=lambda e:e.get("published_date") or "", reverse=True)
    lines, used = [], 0
    for e in edges:
        line = (
            f"{e['source_name']} [{e['source_type']}] -{e['relation']}-> "
            f"{e['target_name']} [{e['target_type']}] "
            f"| date={e.get('published_date') or 'unknown'} "
            f"| chunk={e.get('source_chunk_id') or 'unknown'}"
        )
        if e.get("evidence"):
            line += f" | evidence={norm_space(e['evidence'])}"
        if used + len(line) + 1 > MAX_GRAPH_CONTEXT_CHARS:
            break
        lines.append(line)
        used += len(line) + 1
    return "\n".join(lines)

def retrieve_graph_context(query, max_hops=2, edge_limit=50, return_debug=False):
    seeds = match_seeds(query)
    if not seeds:
        out = {"context":"","edges":pd.DataFrame(),
               "diagnostics":{"reason":"NO_SEED","supernode_events":[]}}
        return out if return_debug else ""

    frontier = deque((x["id"],0) for x in seeds)
    expanded, seen_edges, collected = set(), set(), []
    supernode_events = []

    while frontier and len(collected) < GLOBAL_EDGE_CAP:
        node_id, hop = frontier.popleft()
        if node_id in expanded or hop >= max_hops:
            continue
        expanded.add(node_id)

        degree = node_degree(node_id)
        limit = int(edge_limit)
        if degree > SUPER_NODE_DEGREE:
            limit = min(limit, SUPER_NODE_EDGE_CAP)
            supernode_events.append({"node_id":node_id,"degree":degree,"limit":limit})

        for e in recent_edges(node_id, limit):
            key = (e["source_id"],e["relation"],e["target_id"],e["source_chunk_id"])
            if key in seen_edges:
                continue
            seen_edges.add(key)
            collected.append(e)
            if len(collected) >= GLOBAL_EDGE_CAP:
                break

            nb = e.get("neighbor_id")
            if nb and nb not in expanded and hop + 1 < max_hops:
                frontier.append((nb, hop+1))

    out = {
        "context": textualize(collected),
        "edges": pd.DataFrame(collected),
        "diagnostics": {
            "matched_seeds": seeds,
            "expanded_nodes": len(expanded),
            "collected_edges": len(collected),
            "supernode_events": supernode_events,
        }
    }
    return out if return_debug else out["context"]


In [27]:
ANSWER_SYSTEM = """
Answer only from supplied context.
Be concise but complete. Do not invent facts.
Cite provenance inline as [chunk_id=...] whenever possible.
If evidence is insufficient or conflicting, say so.
""".strip()

def generate_answer(question, context):
    prompt = f"QUESTION:\n{question}\n\nCONTEXT:\n{context}\n\nANSWER:"
    t0 = time.perf_counter()
    text, usage = openai_chat(
        [{"role":"system","content":ANSWER_SYSTEM},
         {"role":"user","content":prompt}],
        model=OPENAI_MODEL
    )
    return {
        "answer": text.strip(),
        "latency_s": time.perf_counter()-t0,
        "total_tokens": usage.get("total_tokens"),
    }

def answer_flat_rag(question):
    context, retrieved = retrieve_flat_context(question, k=6)
    out = generate_answer(question, context)
    out.update({"context":context,"retrieved":retrieved})
    return out

def answer_graph_rag(question):
    g = retrieve_graph_context(question, max_hops=2, edge_limit=50, return_debug=True)
    vctx, vdocs = retrieve_flat_context(question, k=4)
    context = f"=== GRAPH ===\n{g['context']}\n\n=== VECTOR ===\n{vctx}"
    out = generate_answer(question, context)
    out.update({"context":context,"graph_debug":g,"vector_docs":vdocs})
    return out


In [31]:
GOLDEN_PATH = "/content/graphrag_golden_50_first5000_detailed.csv"

starter_golden = pd.DataFrame([
    {
        "id":"G01","group":"factoid",
        "question":"Who was the CEO of Hugging Face in 2023?",
        "reference_answer":"Clément Delangue",
        "reference_evidence":"Validate against instructor dump."
    },
    {
        "id":"G02","group":"multi-hop",
        "question":"Which startups were founded by former Microsoft employees and later received investment from Google?",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
    {
        "id":"G03","group":"cross-doc",
        "question":"Compare the direction of AI-related investments by Meta and Apple during 2023 using evidence from multiple articles.",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
    {
        "id":"G04","group":"multi-hop",
        "question":"Find a company invested in by a major technology company that also developed a named AI technology; identify both relations and dates.",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
    {
        "id":"G05","group":"cross-doc",
        "question":"Identify one technology connected to the same company in at least two news chunks and summarize how the relationship changed over time.",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
])

golden_df = pd.read_csv(GOLDEN_PATH) if Path(GOLDEN_PATH).exists() else starter_golden.copy()
display(golden_df)

def validate_golden(df, require_answers=True):
    required = {"id","group","question","reference_answer"}
    if not required.issubset(df.columns):
        raise ValueError(f"Missing columns: {required-set(df.columns)}")
    if require_answers and df.reference_answer.fillna("").str.strip().eq("").any():
        display(df[df.reference_answer.fillna("").str.strip().eq("")][["id","question"]])
        raise ValueError("Điền reference_answer trước final evaluation.")
    print("✅ Golden Dataset valid.")


,id,group,difficulty,question,reference_answer,reference_evidence,evidence_row_ids_0based,evidence_urls_json,expected_hops,seed_entities,required_relations,adversarial_dimension,gold_reasoning,scoring_notes,source_scope
0,G5000-01,multi-hop,hard,Reconstruct the Aeris–Ericsson IoT transaction across the available reports: which Ericsson businesses moved to Aeri...,"Ericsson's IoT Accelerator and Connected Vehicle Cloud businesses, together with related assets, were to be transfer...",row 33 (2022-12-07 13:45:00): Aeris to Acquire IoT Business from Ericsson | row 1746 (2023-01-10 06:19:00): Aeris to...,"[33, 1746, 935]","[""https://www.benzinga.com/pressreleases/22/12/b29992210/aeris-to-acquire-iot-business-from-ericsson"", ""https://elec...",3,"[""Aeris"", ""Ericsson"", ""IoT Accelerator"", ""Connected Vehicle Cloud""]","[""ACQUIRED"", ""TRANSFERRED_TO"", ""SUPPORTS""]",Temporal state transition + cross-document scale aggregation,"Link the announcement/transfer language in the earlier reports to the later acquired-state report, then aggregate th...",Full credit requires both asset names and all three scale figures; do not treat the three reports as three separate ...,ONLY first 5000 data rows of hackernoon_subset.csv (pandas index 0-4999)
1,G5000-02,cross-doc,hard,"Did the first two Aeris/Ericsson reports describe a completed acquisition or a planned transfer, and what later evid...",The first reports describe a planned transaction: Aeris was to acquire Ericsson's IoT Accelerator and Connected Vehi...,row 33 (2022-12-07 13:45:00): Aeris to Acquire IoT Business from Ericsson | row 1746 (2023-01-10 06:19:00): Aeris to...,"[33, 1746, 935]","[""https://www.benzinga.com/pressreleases/22/12/b29992210/aeris-to-acquire-iot-business-from-ericsson"", ""https://elec...",2,"[""Aeris"", ""Ericsson""]","[""PLANNED_ACQUISITION"", ""ACQUIRED""]",Temporal event-state disambiguation,Compare verbs and dates across the three documents; infer only the state progression explicitly supported by wording.,"Penalize answers that say the acquisition was already completed on Dec. 7 solely because the title says ""to Acquire"".",ONLY first 5000 data rows of hackernoon_subset.csv (pandas index 0-4999)
2,G5000-03,factoid,hard,"After the Aeris–Ericsson IoT deal progressed, how many IoT devices, enterprises, and countries were cited in the lat...","More than 100 million IoT devices, 9,000 enterprises, and 190 countries.",row 935 (2023-01-18 22:37:00): A Leap in Connectivity: Aeris Acquires Technologies from Ericsson to Support Cellular...,[935],"[""https://www.iotevolutionworld.com/iot/articles/454653-leap-connectivity-aeris-acquires-technologies-from-ericsson-...",1,"[""Aeris"", ""Ericsson""]","[""SUPPORTS""]",Numeric precision; hallucination trap,Read all three scale values from the same evidence record.,All three numbers must be correct; no partial invention.,ONLY first 5000 data rows of hackernoon_subset.csv (pandas index 0-4999)
3,G5000-04,cross-doc,hard,"Which two named Ericsson IoT businesses recur across multiple reports of the Aeris transaction, and why should Graph...",The recurring businesses are Ericsson IoT Accelerator and Connected Vehicle Cloud. The reports describe the same Aer...,row 33 (2022-12-07 13:45:00): Aeris to Acquire IoT Business from Ericsson | row 1746 (2023-01-10 06:19:00): Aeris to...,"[33, 1746, 935]","[""https://www.benzinga.com/pressreleases/22/12/b29992210/aeris-to-acquire-iot-business-from-ericsson"", ""https://elec...",2,"[""Aeris"", ""Ericsson"", ""IoT Accelerator"", ""Connected Vehicle Cloud""]","[""ACQUIRED""]",Deduplication + entity/event resolution,Identify repeated assets and connect the reports as syndicated/follow-up coverage of one transaction.,Full credit requires explicit event deduplication reasoning.,ONLY first 5000 data rows of hackernoon_subset.csv (pandas index 0-4999)
4,G5000-05,multi-hop,hard,"Starting from Ericsson, follow the graph to the acquirer and then to the reported IoT reach. What path and scale sh

In [32]:
JUDGE_SYSTEM = """
You are a strict evaluator of RAG answers.
Score 1-5:
- comprehensiveness
- faithfulness to supplied candidate context
- multi_hop_reasoning accuracy
Use the reference answer as correctness anchor.
Return strict JSON only.
""".strip()

def judge_json(system, user):
    if not OPENAI_JUDGE_MODEL:
        raise RuntimeError("Thiếu OPENAI_JUDGE_MODEL.")
    return openai_json(system, user, model=OPENAI_JUDGE_MODEL)[0]

def judge_answer(question, reference, answer, context):
    prompt = f"""
QUESTION:
{question}

REFERENCE:
{reference}

CANDIDATE:
{answer}

CANDIDATE CONTEXT:
{context[:18000]}

Return:
{{
 "comprehensiveness":1,
 "faithfulness":1,
 "multi_hop_reasoning":1,
 "rationale":"2-5 sentences"
}}
"""
    obj = judge_json(JUDGE_SYSTEM, prompt)
    out = {}
    for k in ["comprehensiveness","faithfulness","multi_hop_reasoning"]:
        out[k] = max(1, min(5, int(obj.get(k,1))))
    out["rationale"] = norm_space(obj.get("rationale"))
    return out


In [33]:
CHECKPOINT = "/content/graphrag_eval_checkpoint.csv"

def run_evaluation(golden_df):
    rows = []
    for q in tqdm(golden_df.itertuples(index=False), total=len(golden_df), desc="Evaluation"):
        flat = answer_flat_rag(q.question)
        graph = answer_graph_rag(q.question)

        jf = judge_answer(q.question, q.reference_answer, flat["answer"], flat["context"])
        jg = judge_answer(q.question, q.reference_answer, graph["answer"], graph["context"])

        rows.append({
            "id":q.id, "group":q.group, "question":q.question,
            "reference_answer":q.reference_answer,
            "flat_answer":flat["answer"], "graph_answer":graph["answer"],
            "flat_comprehensiveness":jf["comprehensiveness"],
            "graph_comprehensiveness":jg["comprehensiveness"],
            "flat_faithfulness":jf["faithfulness"],
            "graph_faithfulness":jg["faithfulness"],
            "flat_multi_hop_reasoning":jf["multi_hop_reasoning"],
            "graph_multi_hop_reasoning":jg["multi_hop_reasoning"],
            "flat_latency_s":flat["latency_s"],
            "graph_latency_s":graph["latency_s"],
            "flat_total_tokens":flat.get("total_tokens"),
            "graph_total_tokens":graph.get("total_tokens"),
            "flat_judge_rationale":jf["rationale"],
            "graph_judge_rationale":jg["rationale"],
            "graph_supernode_events":len(
                graph["graph_debug"]["diagnostics"].get("supernode_events",[])
            )
        })
        pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)
    return pd.DataFrame(rows)

validate_golden(golden_df, require_answers=True)
eval_results_df = run_evaluation(golden_df)
display(eval_results_df)


✅ Golden Dataset valid.


Evaluation:   0%|          | 0/50 [00:00<?, ?it/s]

,id,group,question,reference_answer,flat_answer,graph_answer,flat_comprehensiveness,graph_comprehensiveness,flat_faithfulness,graph_faithfulness,flat_multi_hop_reasoning,graph_multi_hop_reasoning,flat_latency_s,graph_latency_s,flat_total_tokens,graph_total_tokens,flat_judge_rationale,graph_judge_rationale,graph_supernode_events
0,G5000-01,multi-hop,Reconstruct the Aeris–Ericsson IoT transaction across the available reports: which Ericsson businesses moved to Aeri...,"Ericsson's IoT Accelerator and Connected Vehicle Cloud businesses, together with related assets, were to be transfer...",The available context does not provide specific details about the Ericsson businesses that moved to Aeris or the sca...,"The available context does not provide specific details about the Aeris–Ericsson IoT transaction, including which Er...",1,1,1,1,1,1,1.313128,1.158335,580,463,The candidate fails to provide any specific details regarding the Ericsson businesses that moved to Aeris or the sca...,"The candidate fails to provide any specific details regarding the Aeris–Ericsson IoT transaction, including which Er...",0
1,G5000-02,cross-doc,"Did the first two Aeris/Ericsson reports describe a completed acquisition or a planned transfer, and what later evid...",The first reports describe a planned transaction: Aeris was to acquire Ericsson's IoT Accelerator and Connected Vehi...,The first two Aeris/Ericsson reports do not provide sufficient information to determine whether they describe a comp...,The first two Aeris/Ericsson reports describe a planned transfer rather than a completed acquisition. There is no sp...,1,1,1,1,1,1,1.264776,1.426313,570,417,"The candidate fails to address the specifics of the Aeris/Ericsson reports, incorrectly stating that there is insuff...","The candidate fails to acknowledge that the first two reports indicated a planned transfer, while the later report c...",0
2,G5000-03,factoid,"After the Aeris–Ericsson IoT deal progressed, how many IoT devices, enterprises, and countries were cited in the lat...","More than 100 million IoT devices, 9,000 enterprises, and 190 countries.","The provided context does not specify the number of IoT devices, enterprises, or countries cited in the later connec...","The provided context does not specify the number of IoT devices, enterprises, and countries cited in the later conne...",1,1,1,1,1,1,0.841402,1.202646,575,440,"The candidate fails to provide any specific information regarding the number of IoT devices, enterprises, or countri...","The candidate fails to provide any specific information regarding the number of IoT devices, enterprises, and countr...",0
3,G5000-04,cross-doc,"Which two named Ericsson IoT businesses recur across multiple reports of the Aeris transaction, and why should Graph...",The recurring businesses are Ericsson IoT Accelerator and Connected Vehicle Cloud. The reports describe the same Aer...,The two named Ericsson IoT businesses that recur across multiple reports of the Aeris transaction are not specified ...,The context provided does not mention any specific Ericsson IoT businesses or details about the Aeris transaction. T...,1,1,1,1,1,1,1.033205,1.141472,482,385,"The candidate fails to identify the two named Ericsson IoT businesses, which are crucial to answering the question. ...",The candidate fails to provide any information regarding the specific Ericsson IoT businesses mentioned in the quest...,0
4,G5000-05,multi-hop,"Starting from Ericsson, follow the graph to the acquirer and then to the reported IoT reach. What path and scale sho...",Ericsson -> (IoT Accelerator and Connected Vehicle Cloud transferred/acquired by) Aeris -> supports/connects more th...,"Starting from Ericsson, the path to the acquirer is not explicitly stated in the provided context. Therefore, I cann...","The path from Ericsson to the acquirer is not explicitly provided in the context. Therefore, I cannot determine the ...",1,1,1,1,1,1,1.722889,1.343581,615,476,"The candidate fails t

In [34]:
def comparison_table(eval_df):
    metric_map = {
        "Comprehensiveness":("flat_comprehensiveness","graph_comprehensiveness"),
        "Faithfulness":("flat_faithfulness","graph_faithfulness"),
        "Multi-hop reasoning":("flat_multi_hop_reasoning","graph_multi_hop_reasoning"),
        "Latency (s)":("flat_latency_s","graph_latency_s"),
        "Token usage":("flat_total_tokens","graph_total_tokens"),
    }

    rows = []
    for group, g in eval_df.groupby("group"):
        for metric, (fc,gc) in metric_map.items():
            f = pd.to_numeric(g[fc], errors="coerce").mean()
            gr = pd.to_numeric(g[gc], errors="coerce").mean()

            if metric in {"Latency (s)","Token usage"}:
                comment = "Flat RAG thường rẻ/nhanh hơn." if f < gr else "GraphRAG không đắt hơn trong sample này."
            else:
                delta = gr - f
                if delta >= .75:
                    comment = "GraphRAG cải thiện rõ; kiểm tra rationale và provenance."
                elif delta <= -.5:
                    comment = "Flat RAG tốt hơn; graph extraction/retrieval có thể gây mất thông tin hoặc nhiễu."
                else:
                    comment = "Hai phương pháp gần nhau."

            rows.append({
                "Loại câu hỏi":group, "Metric":metric,
                "Flat RAG":round(f,3) if pd.notna(f) else np.nan,
                "GraphRAG":round(gr,3) if pd.notna(gr) else np.nan,
                "Nhận xét phân tích":comment
            })
    return pd.DataFrame(rows)

comparison_df = comparison_table(eval_results_df)
display(comparison_df)
eval_results_df.to_csv("/content/graphrag_eval_results.csv", index=False)
comparison_df.to_csv("/content/graphrag_vs_flatrag_summary.csv", index=False)


,Loại câu hỏi,Metric,Flat RAG,GraphRAG,Nhận xét phân tích
0,cross-doc,Comprehensiveness,2.727,2.591,Hai phương pháp gần nhau.
1,cross-doc,Faithfulness,3.136,2.909,Hai phương pháp gần nhau.
2,cross-doc,Multi-hop reasoning,2.727,2.591,Hai phương pháp gần nhau.
3,cross-doc,Latency (s),1.681,1.674,GraphRAG không đắt hơn trong sample này.
4,cross-doc,Token usage,627.955,575.091,GraphRAG không đắt hơn trong sample này.
5,factoid,Comprehensiveness,2.600,2.000,Flat RAG tốt hơn; graph extraction/retrieval có thể gây mất thông tin hoặc nhiễu.
6,factoid,Faithfulness,2.600,2.000,Flat RAG tốt hơn; graph extraction/retrieval có thể gây mất thông tin hoặc nhiễu.
7,factoid,Multi-hop reasoning,2.600,2.000,Flat RAG tốt hơn; graph extraction/retrieval có thể gây mất thông tin hoặc nhiễu.
8,factoid,Latency (s),0.986,1.031,Flat RAG thường rẻ/nhanh hơn.
9,factoid,Token usage,581.400,507.800,GraphRAG không đắt hơn trong sample này.


In [35]:
def test_supernode_policy():
    rows = run_cypher("""
    MATCH (n:Entity)-[r]-()
    WITH n, count(r) AS degree
    ORDER BY degree DESC LIMIT 1
    RETURN n.id AS id, n.name AS name, degree
    """)
    if not rows:
        print("Graph empty.")
        return

    n = rows[0]
    limit = 50 if n["degree"] > SUPER_NODE_DEGREE else 1000
    edges = recent_edges(n["id"], limit)
    print(n, "fetched=", len(edges))
    if n["degree"] > SUPER_NODE_DEGREE:
        assert len(edges) <= 50
        print("✅ Super-node cap OK.")

def show_resolution_audit(audit_df):
    if audit_df.empty:
        print("No audit rows.")
        return
    display(
        audit_df.sort_values("similarity", ascending=False).head(30)
    )
    print("High-similarity rejected pairs:")
    display(
        audit_df[audit_df.decision=="REJECT_GUARD"]
        .sort_values("similarity", ascending=False)
        .head(20)
    )

test_supernode_policy()
show_resolution_audit(entity_resolution_audit_df)


{'id': '482e949a0b6c21ca2b8da03e', 'name': 'Infineon', 'degree': 3} fetched= 3


,type,left,right,similarity,decision
2,Company,Fidelity National Information Services,Fidelity National Information Services Inc.,0.982646,MERGE_VECTOR
3,Company,Biz Technology Solutions,Biz Technology Solutions Inc.,0.951118,MERGE_VECTOR
1,Company,Amazon Web Services (AWS),Amazon Web Services,0.941637,MERGE_VECTOR
0,Company,Information Services Group,Information Services Group Inc.,0.935348,MERGE_VECTOR
4,Technology,X90A,X90,0.932864,MERGE_VECTOR


High-similarity rejected pairs:


,type,left,right,similarity,decision


In [40]:
import networkx as nx

def build_communities(limit_edges=20000):
    edge_df = pd.DataFrame(run_cypher("""
    MATCH (a:Entity)-[r]->(b:Entity)
    RETURN a.id AS source, b.id AS target
    LIMIT $limit
    """, limit=int(limit_edges)))

    G = nx.Graph()
    G.add_edges_from(edge_df[["source","target"]].itertuples(index=False, name=None))
    communities = nx.algorithms.community.greedy_modularity_communities(G)

    rows = []
    for cid, members in enumerate(communities):
        rows += [{"id":node_id,"community_id":int(cid)} for node_id in members]

    for b in batches(rows, 1000):
        run_cypher("""
        UNWIND $rows AS row
        MATCH (n:Entity {id:row.id})
        SET n.community_id=row.community_id
        """, rows=b)

    return pd.DataFrame(rows)

community_df = build_communities()
display(community_df)
community_df["community_id"].value_counts().head(20)

,id,community_id
0,ad7c2c62d5269906c4019b77,0
1,0cf2f2945e5f6dc338954501,0
2,5b287402f12760f174928569,0
3,a13db9241151b8c4fe050342,0
4,b2d4ccacccbb87fe51c0d279,1
...,...,...
281,5f4c36a21d6cb0912c521527,123
282,7661cb417d232058bc65aa7b,124
283,ec0b2fdb9810b5d6d2a42951,124
284,4ae8e5bf1558fd593bd0e078,125


,count
community_id,
0,4
1,4
2,4
3,4
4,4
5,3
6,3
7,3
8,3


In [37]:
SUFFICIENCY_SYSTEM = """
Decide whether the supplied retrieval context is sufficient to answer the question faithfully.
Do not answer the question. Return strict JSON only.
""".strip()

def context_sufficient(question, context):
    obj, _ = openai_json(
        SUFFICIENCY_SYSTEM,
        f"""QUESTION: {question}
CONTEXT:
{context[:16000]}
Return {{"sufficient":true,"missing":"..."}}"""
    )
    return bool(obj.get("sufficient")), norm_space(obj.get("missing"))

def self_correcting_context(question):
    g2 = retrieve_graph_context(question, 2, 50, True)
    ok, missing = context_sufficient(question, g2["context"])
    if ok:
        return {"route":"hop2","context":g2["context"],"missing":""}

    g3 = retrieve_graph_context(question, 3, 50, True)
    ok, missing2 = context_sufficient(question, g3["context"])
    if ok:
        return {"route":"hop3","context":g3["context"],"missing":missing}

    flat, _ = retrieve_flat_context(question, k=8)
    return {
        "route":"hop3+vector",
        "context":f"=== GRAPH ===\n{g3['context']}\n\n=== VECTOR ===\n{flat}",
        "missing":missing2
    }


In [41]:
question = "Who was the CEO of Hugging Face in 2023?"

graph_result = retrieve_graph_context(
    question,
    max_hops=2,
    edge_limit=50,
    return_debug=True
)

context = graph_result["context"]

sufficient, missing = context_sufficient(question, context)

print("Sufficient:", sufficient)
print("Missing:", missing)

Sufficient: False
Missing: The name of the CEO of Hugging Face in 2023.


In [42]:
result = self_correcting_context(
    "Which companies were founded by former Microsoft employees and later received investment from Google?"
)

print("Route:", result["route"])
print("Missing:", result["missing"])
print("\nContext preview:\n")
print(result["context"][:3000])

Route: hop3+vector
Missing: Specific names of companies founded by former Microsoft employees that received investment from Google.

Context preview:

=== GRAPH ===


=== VECTOR ===
[chunk_id=19320fc9b3bfcf0ed531::c0000 | date=2022-12-28 | score=0.466]
Years later and Russian tech companies have grown their footprint. Google GOOG came to town but was quickly beaten out of the market by Yandex once tradable on the Nasdaq. Now like all things Russian

[chunk_id=0611dcc975c2cc291a1f::c0000 | date=2023-01-19 | score=0.455]
Last year was terrible for some of the best cloud computing stocks and for the Cloud Czars whose cash flow built it. Apple (NASDAQ:AAPL) Microsoft (NASDAQ:MSFT) Alphabet (NASDAQ:GOOGL GOOG) Amazon.

[chunk_id=af275a0ced2573ae53ba::c0000 | date=2023-03-01 | score=0.441]
Microsoft announced its $1 billion Climate Innovation Fund and six months later Amazon unveiled its $2 billion Climate Pledge Fund. An analysis of the investments made since… Read More

[chunk_id=c833db37e